In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import torch
import random

from datasets import load_dataset

from decider.model import DecisionModel

/Users/sam/Desktop/projects/llm/llms/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1025 23:51:14.246000 55965 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
seed: int = 42
model_id: str = "Qwen/Qwen3.5-0.8B"
revision: str = "2b48083dfa97c3cbf0220cf1a5e6cffe0a511157"
dataset_id: str = "LocalLLaMA/typed-decisions"
dataset_name: str | None = "all"
dataset_split: str = "train"
dataset_streaming: bool = False
max_seq_len: int = 2048


In [4]:
random.seed(seed)
torch.manual_seed(seed)

In [5]:
model = DecisionModel(
    model_id=model_id,
    revision=revision,
    device=torch.device("mps"),
)

tokenizer = model.tokenizer
processor = model.processor
decision_codes = model.decision_codes

Loading weights: 100%|██████████| 473/473 [00:00<00:00, 7783.29it/s]


In [6]:
dataset_id = dataset_id
dataset_kwargs = {
	"name": dataset_name,
	"streaming": dataset_streaming,
	"split": dataset_split,
}
dataset = load_dataset(dataset_id, **dataset_kwargs)

In [7]:
import json

def parse(value):
    try:
        return json.loads(value)
    except:
        return value

In [8]:
import math

from decider.types import DecisionInput

row = next(iter(dataset))

questions = parse(row["questions"])
state = parse(row["state"])
gold = parse(row["gold"])

batch_input = DecisionInput(questions=questions, state=state, gold=gold)

batch = model.encoder.prepare_batch(
    rows=[batch_input],
    max_length=256,
)

for chunk in batch:
	assert chunk.targets is not None
	assert chunk.labels is not None

for chunk in batch:
	for question_key, option_keys, target, label in zip(
		chunk.keys,
		chunk.option_keys,
		chunk.targets,
		chunk.labels,
		strict=True,
	):
		answer = gold[question_key]

		expected_target = tuple(
			float(answer["probabilities"][option_key]) for option_key in option_keys
		)

		assert all(
			math.isclose(
				actual,
				expected,
				rel_tol=0.0,
				abs_tol=1e-7,
			)
			for actual, expected in zip(
				target,
				expected_target,
				strict=True,
			)
		)

		expected_label = answer["label"]

		if questions[question_key]["type"] == "noul":
			expected_label = expected_label.lower()

		assert option_keys[label] == expected_label

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


In [9]:
chunks = batch

outputs = model(chunks)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


In [10]:
from decider.loss import decision_loss


components = decision_loss(
    outputs=outputs,
    chunks=chunks,
    brier_weight=0.0,
)

assert components.total.ndim == 0
assert components.cross_entropy.ndim == 0
assert components.brier.ndim == 0

assert torch.isfinite(components.total)
assert torch.isfinite(components.cross_entropy)
assert torch.isfinite(components.brier)

print("total:", components.total.item())
print("cross_entropy:", components.cross_entropy.item())
print("brier:", components.brier.item())

total: 1.858246088027954
cross_entropy: 1.858246088027954
brier: 0.3374170660972595


In [11]:
components.total.backward()

In [12]:
model.zero_grad(set_to_none=True)

outputs = model(chunks)

components = decision_loss(
    outputs=outputs,
    chunks=chunks,
    brier_weight=0.0,
)

components.total.backward()

readout_grad = model.readout.weight.grad

assert readout_grad is not None
assert torch.isfinite(readout_grad).all()
assert readout_grad.abs().sum().item() > 0

print("Readout gradient check passed")
print("Gradient norm:", readout_grad.norm().item())

Readout gradient check passed
Gradient norm: 54.75


In [13]:
for name, parameter in model.backbone.named_parameters():
    assert parameter.requires_grad is False
    assert parameter.grad is None

print("Backbone freeze check passed")

Backbone freeze check passed
